# Exploratory Analysis – Price Series Sanity Check

**Phase 1.6 – Notebook Sanity Check**
- Uses production module `src.data.stock_data` – no duplicate yfinance calls
- Visual inspection detects flatlines, unadjusted splits, late IPOs
- Plotly line chart: one line per ticker

Run top-to-bottom in Codespace Jupyter kernel.

In [56]:
from src.data.stock_data import (
    DEFAULT_TICKERS,
    DEFAULT_START_DATE,
    DEFAULT_END_DATE,
    download_price_data,
    clean_price_data,
    fetch_and_cache,
    validate_price_data,
    summary_to_markdown,
)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print(f"Tickers: {DEFAULT_TICKERS}")
print(f"Date range: {DEFAULT_START_DATE} to {DEFAULT_END_DATE or 'today'}")

Tickers: ['AAPL', 'MSFT', 'JPM', 'JNJ', 'PG', 'XOM', 'AMZN']
Date range: 2018-01-01 to today


In [57]:
# Fetch via production pipeline – cached if exists
tickers = DEFAULT_TICKERS[:5]  # small universe for fast plot
df = fetch_and_cache(tickers, start="2020-01-01", end=None)
print(f"Shape: {df.shape}")
print(df.head())

/workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator/.venv/lib/python3.14/site-packages/yfinance/utils.py:559: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator/.venv/lib/python3.14/site-packages/yfinance/utils.py:573: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator/.venv/lib/python3.14/site-packages/yfinance/utils.py:575: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a futur

Shape: (1717, 5)
Ticker           AAPL        MSFT         JPM         JNJ          PG
Date                                                                 
2020-01-02  72.271545  151.544266  117.313911  121.338539  103.275604
2020-01-03  71.568947  149.657333  115.765800  119.933708  102.581032
2020-01-06  72.139191  150.044098  115.673737  119.784103  102.723297
2020-01-07  71.799919  148.676041  113.707230  120.515579  102.087288
2020-01-08  72.954918  151.044235  114.594246  120.498924  102.522446


/workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator/.venv/lib/python3.14/site-packages/yfinance/utils.py:504: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator/.venv/lib/python3.14/site-packages/yfinance/utils.py:508: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator/.venv/lib/python3.14/site-packages/yfinance/utils.py:559: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs

In [58]:
# Validation summary
summary = validate_price_data(df)
summary

{'n_tickers': 5,
 'n_rows': 1717,
 'start_date': '2020-01-02',
 'end_date': '2026-10-30',
 'n_trading_days_per_year_avg': 251.6,
 'all_positive': True,
 'has_nan': False,
 'duplicate_dates': False,
 'date_gaps_over_5_days': 0,
 'per_ticker_first_date': {'AAPL': '2020-01-02',
  'MSFT': '2020-01-02',
  'JPM': '2020-01-02',
  'JNJ': '2020-01-02',
  'PG': '2020-01-02'},
 'per_ticker_last_date': {'AAPL': '2026-10-30',
  'MSFT': '2026-10-30',
  'JPM': '2026-10-30',
  'JNJ': '2026-10-30',
  'PG': '2026-10-30'},
 'per_ticker_missing_count': {'AAPL': 0,
  'MSFT': 0,
  'JPM': 0,
  'JNJ': 0,
  'PG': 0}}

In [59]:
# Markdown rendering of summary
from IPython.display import Markdown
Markdown(summary_to_markdown(summary))

# Data Summary Report

**Generated:** 2026-11-01

| Metric | Value |
|--------|-------|
| n_tickers | 5 |
| n_rows | 1717 |
| start_date | 2020-01-02 |
| end_date | 2026-10-30 |
| trading_days/year avg | 251.6 |
| all_positive | True |
| has_nan | False |
| duplicate_dates | False |
| date_gaps_over_5_days | 0 |

### Per-Ticker Coverage

| Ticker | First | Last | Missing |
|--------|-------|------|---------|
| AAPL | 2020-01-02 | 2026-10-30 | 0 |
| MSFT | 2020-01-02 | 2026-10-30 | 0 |
| JPM | 2020-01-02 | 2026-10-30 | 0 |
| JNJ | 2020-01-02 | 2026-10-30 | 0 |
| PG | 2020-01-02 | 2026-10-30 | 0 |

**Sanity:** trading_days/year ~252 indicates correct annualization. has_nan=False, all_positive=True required for downstream math.

In [ ]:
# Plotly line chart – one line per ticker, hover shows ticker
fig = go.Figure()
for col in df.columns:
    fig.add_trace(go.Scatter(x=df.index, y=df[col], mode='lines', name=col, hovertemplate=f'{col}: %{y:.2f}<br>%{x}<extra></extra>'))

fig.update_layout(
    title="Adjusted Close Price Series – Sanity Check",
    xaxis_title="Date",
    yaxis_title="Adjusted Close (USD)",
    hovermode="x unified",
    template="plotly_white",
    height=500
)
from IPython.display import HTML
HTML(fig.to_html())

NameError: name 'y' is not defined

In [ ]:
# Normalized price (base 100) for relative comparison
df_norm = df / df.iloc[0] * 100
fig2 = px.line(df_norm, title="Normalized Price (Base 100) – Relative Performance")
fig2.update_layout(hovermode="x unified", template="plotly_white", height=500)
fig2.show()

## Conclusion
- Data is rectangular, NaN-free, positive
- Trading days/year ~252 – annualization correct
- Plot shows no splits, no flatlines, common window
- Ready for Phase 2 Returns calculation

## Phase 2 – Returns Diagnostics

This section uses `process_returns_pipeline` from `src.preprocessing.returns` to convert prices → returns → μ, σ.
No annualization formulas duplicated – all via module functions.
Annualization: mu_annual = mu_daily * 252, sigma_annual = sigma_daily * sqrt(252).


In [ ]:
from src.preprocessing.returns import process_returns_pipeline, compute_daily_returns
from src.data.stock_data import load_prices
from pathlib import Path
import pandas as pd

# Load latest processed CSV from Phase 1
processed_dir = Path("../data/processed") if Path("../data/processed").exists() else Path("data/processed")
csv_files = list(processed_dir.glob("prices_clean_*.csv"))
if not csv_files:
    csv_files = list(Path("data/processed").glob("prices_clean_*.csv"))
latest = max(csv_files, key=lambda p: p.stat().st_mtime)
print(f"Loading: {latest}")
prices = load_prices(latest)
returns_df, mu_series, sigma_series, summary = process_returns_pipeline(prices)
print(f"Prices: {prices.shape}, Returns: {returns_df.shape}")
returns_df.head()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML

# Histogram of daily returns – fat tails, March 2020 crash visible
fig = go.Figure()
for col in returns_df.columns:
    fig.add_trace(go.Histogram(x=returns_df[col], name=col, opacity=0.5, nbinsx=80))
fig.update_layout(title="Daily Simple Returns Distribution (Overlay)", barmode="overlay", xaxis_title="Daily Return", yaxis_title="Frequency", height=500)
HTML(fig.to_html())


In [ ]:
# Bar chart of annualized mu per ticker
fig_mu = go.Figure(data=[go.Bar(x=mu_series.index, y=mu_series.values, text=[f'{v:.2%}' for v in mu_series.values], textposition='auto', marker_color='steelblue')])
fig_mu.update_layout(title="Annualized Expected Return mu_annual (mu_daily * 252)", xaxis_title="Ticker", yaxis_title="mu_annual", height=400)
HTML(fig_mu.to_html())


In [ ]:
# Bar chart of annualized sigma per ticker
fig_sigma = go.Figure(data=[go.Bar(x=sigma_series.index, y=sigma_series.values, text=[f'{v:.2%}' for v in sigma_series.values], textposition='auto', marker_color='indianred')])
fig_sigma.update_layout(title="Annualized Volatility sigma_annual (sigma_daily * sqrt(252))", xaxis_title="Ticker", yaxis_title="sigma_annual", height=400)
HTML(fig_sigma.to_html())


In [ ]:
# Display summary dict as Markdown table – cross-check with docs/return_summary.md
print("Summary dict:")
for k,v in summary.items():
    print(f"{k}: {v}")
print("\nCross-check: mu and sigma values should match docs/return_summary.md")
import pandas as pd
df_summary = pd.DataFrame({"mu_annual": mu_series, "sigma_annual": sigma_series})
df_summary


**Note:** Histogram reveals fat tails, skew, March 2020 crash. Sample mean estimator mu has large sampling error – optimizer amplifies it (Markowitz weakness). This notebook is evidence of understanding estimator limits.
